In [1]:
from Optomechanics.device_control.Lakeshore_372_AC_Res_Temp_Controller import Temp_Controller

In [2]:
Address =  'GPIB0::12::INSTR' 
#Add comands: temperature mixing champer, temperature 1k pot, voltage or power applyed for the resistance measurment, voltage/current apply to the heater
#still output, PID settings 

with Temp_Controller(Address, ) as TempController:
    print(f"{TempController.IDN_query()}")
    print(f"{TempController.IDN_query()== "LSCI,MODEL372,LSA24XI,1.4\r\n"} ")
    print(TempController.ReadTemperature(3))
    print(TempController.ReadMCTemperature())
    print(TempController.ReadExcitationPower(3))
    print(TempController.ReadResistance(3))  
    print(TempController.ReadInputSetup(3))
    print(TempController.ReadHeaterOutput(3))
    print(TempController.ReadManualHeaterOutput(3))
    print(TempController.ReadHeaterRange(3))
    print(TempController.ReadStillOutput(3))
    print(TempController.ReadPID(3))
    pass

LSCI,MODEL372,LSA24XI,1.4

True 
+293.387E+00

+0.00000E+00

+375.533E-12

+37.6075E+00

0,05,1,10,0,1

+00.000

+000.00

0

+10.00

010.000,00020.0,0000000

device closed


In [ ]:
import time
import datetime
from Optomechanics.device_control.Lakeshore_372_AC_Res_Temp_Controller import Temp_Controller
import os

# Configuration
address = 'GPIB0::12::INSTR'
channel_1k = 3      # Channel for 1K pot thermometer
channel_mc = 4      # Channel for Mixing Chamber thermometer
sample_interval = 1.0  # Time between readings in seconds

def create_filename():
    """Creates a filename with current date and time and full path"""
    
    # Define the directory path
    base_path = r"D:\Python\Optomechanics\Optomechanics\GUI\test_data"
    
    # Create the directory if it doesn't exist
    os.makedirs(base_path, exist_ok=True)
    
    # Create filename with timestamp
    now = datetime.datetime.now()
    filename = f"lakeshore_data_{now.strftime('%Y-%m-%d_%H-%M-%S')}.txt"
    
    # Combine path and filename
    full_path = os.path.join(base_path, filename)
    
    return full_path

def write_header(file):
    """Writes column headers to the file"""
    header = (
        "# Lakeshore 372 Data Log\n"
        f"# Started: {datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n"
        "#\n"
        "# Columns:\n"
        "# Timestamp | T_1k[K] | R_1k[Ohm] | Excitation_1k[W] | T_mc[K] | R_mc[Ohm] | Excitation_mc[W]\n"
        "#\n"
    )
    file.write(header)

def main():
    """Main data logging loop"""
    
    # Create file with timestamp
    filename = create_filename()
    print(f"Starting data logging to: {filename}")
    print(f"Press Ctrl+C to stop\n")
    
    with open(filename, 'w') as datafile:
        # Write header
        write_header(datafile)
        
        # Connect to Lakeshore
        with Temp_Controller(address) as lakeshore:
            
            # Verify connection
            idn = lakeshore.IDN_query()
            print(f"Connected to: {idn}")
            datafile.write(f"# Device: {idn}\n#\n")
            
            # Data logging loop
            try:
                while True:
                    # Get current time
                    timestamp = time.time()
                    
                    # Read 1K pot data
                    t_1k = lakeshore.ReadTemperature(channel_1k)
                    r_1k = lakeshore.ReadResistance(channel_1k)
                    excitation_1k = lakeshore.ReadExcitationPower(channel_1k)
                    
                    # Read Mixing Chamber data
                    t_mc = lakeshore.ReadMCTemperature()  # or ReadTemperature(channel_mc)
                    r_mc = lakeshore.ReadResistance(channel_mc)
                    excitation_mc = lakeshore.ReadExcitationPower(channel_mc)
                    
                    # Format data line
                    data_line = (
                        f"{timestamp:.3f}\t"
                        f"{t_1k:.6f}\t{r_1k:.3f}\t{excitation_1k:.3e}\t"
                        f"{t_mc:.6f}\t{r_mc:.3f}\t{excitation_mc:.3e}\n"
                    )
                    
                    # Write to file
                    datafile.write(data_line)
                    datafile.flush()  # Ensure data is written immediately
                    
                    # Print to console
                    print(f"T_1k: {t_1k:.6f} K | T_mc: {t_mc:.6f} K", end='\r')
                    
                    # Wait for next sample
                    time.sleep(sample_interval)
                    
            except KeyboardInterrupt:
                print("\n\nData logging stopped by user")
                print(f"Data saved to: {filename}")

if __name__ == "__main__":
    main()